# 02 • Selecionar, filtrar e ordenar

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series


### Objetivo: transformar uma pergunta em seleção de dados

Nesta aula você vai responder “quais animais são da ilha Dream?” e “quais têm massa acima de 4.000 g?”. Primeiro construiremos cada condição separadamente; só depois combinaremos condições.

### 1. Comparar uma coluna inteira


In [3]:
print(p['island'].head(3))
condicao = p['island'] == 'Dream'
print(condicao.head(3))


0    Torgersen
1    Torgersen
2    Torgersen
Name: island, dtype: str
0    False
1    False
2    False
Name: island, dtype: bool


`==` pergunta se dois valores são iguais. É diferente de `=`, que guarda um resultado em uma variável. Como a coluna tem vários valores, pandas compara cada um com Dream. Para as primeiras três linhas, o resultado é False, False, False, porque elas pertencem a Torgersen. True significa verdadeiro; False significa falso. A sequência inteira de respostas é chamada máscara booleana.


In [4]:
dream = p.loc[condicao]
print(dream.head(3))


   species island  bill_length_mm  ...  body_mass_g     sex  year
30  Adelie  Dream            39.5  ...       3250.0  female  2007
31  Adelie  Dream            37.2  ...       3900.0    male  2007
32  Adelie  Dream            39.5  ...       3300.0  female  2007

[3 rows x 8 columns]


Leia: “na tabela p, mantenha as linhas em que condicao é verdadeira”. As primeiras linhas do resultado têm índices 30, 31 e 32. Isso não significa que existem 30 linhas escondidas antes da primeira linha exibida: os rótulos originais foram preservados.

### 2. Selecionar linhas e colunas na mesma operação


In [5]:
print(p.loc[condicao, ['species', 'island', 'body_mass_g']].head(3))


   species island  body_mass_g
30  Adelie  Dream       3250.0
31  Adelie  Dream       3900.0
32  Adelie  Dream       3300.0


Dentro de loc, a parte antes da vírgula escolhe linhas; a parte depois escolhe colunas. Não confunda a vírgula que separa os dois critérios com as vírgulas dentro da lista de nomes. Se quiser todas as linhas de certas colunas, use `p.loc[:, ['species', 'island']]`: os dois-pontos significam “todas” nessa posição.

### 3. Condições numéricas e combinações


In [6]:
da_ilha = p['island'] == 'Dream'
pesados = p['body_mass_g'] > 4000
selecao = da_ilha & pesados
print(p.loc[selecao, ['species', 'island', 'body_mass_g']].head())


   species island  body_mass_g
35  Adelie  Dream       4150.0
39  Adelie  Dream       4650.0
43  Adelie  Dream       4400.0
45  Adelie  Dream       4600.0
49  Adelie  Dream       4150.0


O símbolo `&` exige que as duas respostas sejam verdadeiras na mesma linha. Uma massa acima de 4.000 fora de Dream não passa. Um animal de Dream com massa de 3.500 também não passa. Só a interseção passa.

Também podemos escrever sem as variáveis intermediárias, mas mantendo os parênteses de cada comparação:


In [7]:
print(p.loc[(p['island'] == 'Dream') & (p['body_mass_g'] > 4000)].head())


   species island  bill_length_mm  ...  body_mass_g   sex  year
35  Adelie  Dream            39.2  ...       4150.0  male  2007
39  Adelie  Dream            39.8  ...       4650.0  male  2007
43  Adelie  Dream            44.1  ...       4400.0  male  2007
45  Adelie  Dream            39.6  ...       4600.0  male  2007
49  Adelie  Dream            42.3  ...       4150.0  male  2007

[5 rows x 8 columns]


Use `|` para “uma condição ou a outra”, e `~` para inverter uma condição. Não use and/or do Python entre Series; eles não fazem essa combinação linha por linha. Comparações com massa ausente não selecionam uma observação como “pesada”; a aula seguinte mostrará como contar essas ausências.

### 4. A mesma comparação com outros nomes

Você encontrará duas formas de escrever as mesmas operações. Comece pelos símbolos e consulte a tabela quando ler os gabaritos:

| Operação | Símbolo | Método equivalente |
|---|---|---|
| Igual | == | eq(valor) |
| Diferente | != | ne(valor) |
| Maior | > | gt(valor) |
| Maior ou igual | >= | ge(valor) |
| Menor | < | lt(valor) |
| Menor ou igual | <= | le(valor) |

Assim, `p['body_mass_g'].gt(4000)` produz a mesma condição que `p['body_mass_g'] > 4000` nesta base. `isin(['Adelie', 'Chinstrap'])` pergunta se cada espécie pertence à lista informada. É uma maneira de aceitar várias opções para uma coluna.

### 5. Posição não é rótulo


In [8]:
print(dream.iloc[:2][['species', 'island']])
print(dream.loc[30, ['species', 'island']])


   species island
30  Adelie  Dream
31  Adelie  Dream
species    Adelie
island      Dream
Name: 30, dtype: str


iloc usa a posição no resultado: `:2` pega as primeiras duas linhas, nas posições 0 e 1. loc usa o rótulo: 30 identifica uma linha que preservou esse índice da tabela original. Portanto, a primeira linha de dream está na posição zero, mas tem rótulo 30.

Nas fatias por posição, o limite final fica fora: `p.iloc[10:15]` pega as posições 10, 11, 12, 13 e 14. Não tente decorar isso ao mesmo tempo que filtros complexos; teste uma seleção pequena e conte suas linhas. Uma fatia por rótulos em loc pode incluir o limite final.

### 6. Ordenar e pegar os maiores


In [9]:
ordenados = p.sort_values('flipper_length_mm', ascending=False)
print(ordenados[['species', 'flipper_length_mm']].head(5))


    species  flipper_length_mm
215  Gentoo              231.0
265  Gentoo              230.0
267  Gentoo              230.0
217  Gentoo              230.0
227  Gentoo              230.0


sort_values reorganiza linhas pelo valor de uma coluna. ascending=False pede ordem decrescente. head(5) pega as primeiras cinco linhas dessa tabela ordenada. Cada linha continua inteira: a espécie acompanha a medição do mesmo animal. A operação devolve uma nova tabela; o código acima não substitui p.

### Cuidado ao alterar dados

Nesta aula só consultamos dados. Quando precisar alterar, evite uma seleção seguida de outra como `p[condicao]['coluna'] = valor`. Use `p.loc[condicao, 'coluna'] = valor` para uma alteração explícita. Faremos criação de colunas na aula 4. Primeiro garanta que sabe quais linhas a condição seleciona.


## Exemplo guiado

In [10]:
mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen


## Exercício 02.1
Liste apenas pinguins da ilha Dream.

In [11]:
print(p.loc[p['island'].eq('Dream')])

       species island  bill_length_mm  ...  body_mass_g     sex  year
30      Adelie  Dream            39.5  ...       3250.0  female  2007
31      Adelie  Dream            37.2  ...       3900.0    male  2007
32      Adelie  Dream            39.5  ...       3300.0  female  2007
33      Adelie  Dream            40.9  ...       3900.0    male  2007
34      Adelie  Dream            36.4  ...       3325.0  female  2007
..         ...    ...             ...  ...          ...     ...   ...
339  Chinstrap  Dream            55.8  ...       4000.0    male  2009
340  Chinstrap  Dream            43.5  ...       3400.0  female  2009
341  Chinstrap  Dream            49.6  ...       3775.0    male  2009
342  Chinstrap  Dream            50.8  ...       4100.0    male  2009
343  Chinstrap  Dream            50.2  ...       3775.0  female  2009

[124 rows x 8 columns]


## Exercício 02.2
Selecione species, sex e body_mass_g, nessa ordem.

In [12]:
print(p[['species','sex','body_mass_g']].head())

  species     sex  body_mass_g
0  Adelie    male       3750.0
1  Adelie  female       3800.0
2  Adelie  female       3250.0
3  Adelie     NaN          NaN
4  Adelie  female       3450.0


## Exercício 02.3
Encontre animais Adelie ou Chinstrap com massa de pelo menos 4.000 g.

In [13]:
print(p.loc[p['species'].isin(['Adelie','Chinstrap']) & p['body_mass_g'].ge(4000)])

       species     island  bill_length_mm  ...  body_mass_g     sex  year
7       Adelie  Torgersen            39.2  ...       4675.0    male  2007
9       Adelie  Torgersen            42.0  ...       4250.0     NaN  2007
14      Adelie  Torgersen            34.6  ...       4400.0    male  2007
17      Adelie  Torgersen            42.5  ...       4500.0    male  2007
19      Adelie  Torgersen            46.0  ...       4200.0    male  2007
35      Adelie      Dream            39.2  ...       4150.0    male  2007
39      Adelie      Dream            39.8  ...       4650.0    male  2007
43      Adelie      Dream            44.1  ...       4400.0    male  2007
45      Adelie      Dream            39.6  ...       4600.0    male  2007
49      Adelie      Dream            42.3  ...       4150.0    male  2007
51      Adelie     Biscoe            40.1  ...       4300.0    male  2008
53      Adelie     Biscoe            42.0  ...       4050.0    male  2008
61      Adelie     Biscoe            4

## Exercício 02.4
Mostre os cinco animais com maior comprimento de nadadeira.

In [14]:
print(p.sort_values('flipper_length_mm', ascending=False).head(5))

    species  island  bill_length_mm  ...  body_mass_g   sex  year
215  Gentoo  Biscoe            54.3  ...       5650.0  male  2008
265  Gentoo  Biscoe            51.5  ...       5500.0  male  2009
267  Gentoo  Biscoe            55.1  ...       5850.0  male  2009
217  Gentoo  Biscoe            49.8  ...       5700.0  male  2008
227  Gentoo  Biscoe            48.6  ...       5800.0  male  2008

[5 rows x 8 columns]


## Exercício 02.5
Pegue as posições 10 a 14, inclusive. Explique por que o limite usado é 15.

In [15]:
print(p.iloc[10:15])
# O limite superior de iloc fica fora da seleção.

   species     island  bill_length_mm  ...  body_mass_g     sex  year
10  Adelie  Torgersen            37.8  ...       3300.0     NaN  2007
11  Adelie  Torgersen            37.8  ...       3700.0     NaN  2007
12  Adelie  Torgersen            41.1  ...       3200.0  female  2007
13  Adelie  Torgersen            38.6  ...       3800.0    male  2007
14  Adelie  Torgersen            34.6  ...       4400.0    male  2007

[5 rows x 8 columns]
